# Project: HNSW Performance Benchmarking

## Mission

Build on your Day 1 search engine by adding performance optimization. You’ll discover which HNSW settings work best for your specific data and queries, and measure the real impact of payload indexing.

## What we are Building

A performance-optimized version of your Day 1 search engine that demonstrates:

- Fast bulk load: Load with m=0, then switch to HNSW
- HNSW parameter tuning: Try different m and ef_construct
- Payload indexing impact: Time filtering with and without indexes
- Domain findings: What works best for your content

## Step 1: Extend Your Day 1 Project

In [1]:
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer
import time
import numpy as np
import os

client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"))

# For Colab:
# from google.colab import userdata
# client = QdrantClient(url=userdata.get("QDRANT_URL"), api_key=userdata.get("QDRANT_API_KEY"))

encoder = SentenceTransformer("all-MiniLM-L6-v2")

c:\Users\Praneeth\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7190.28it/s]


## Step 2: Create Multiple Test Collections

In [2]:
# Test configurations
configs = [
    {"name": "fast_initial_upload", "m": 0, "ef_construct": 100},  # m=0 = ingest-only
    {"name": "memory_optimized", "m": 8, "ef_construct": 100},  # m=8 = lower RAM
    {"name": "balanced", "m": 16, "ef_construct": 200},  # m=16 = balanced
    {"name": "high_quality", "m": 32, "ef_construct": 400},  # m=32 = higher recall, slower build
]

for config in configs:
    collection_name = f"my_domain_{config['name']}"
    if client.collection_exists(collection_name=collection_name):
        client.delete_collection(collection_name=collection_name)

    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE),
        hnsw_config=models.HnswConfigDiff(
            m=config["m"],
            ef_construct=config["ef_construct"],
            full_scan_threshold=10,  # force HNSW instead of full scan
        ),
        optimizers_config=models.OptimizersConfigDiff(
            indexing_threshold=10
        ),  # Force indexing even on small sets for demo
    )
    print(f"Created collection: {collection_name}")

Created collection: my_domain_fast_initial_upload
Created collection: my_domain_memory_optimized
Created collection: my_domain_balanced
Created collection: my_domain_high_quality


## Step 3: Upload and Time

In [4]:
product_dataset = [
    {"description": "Wireless Bluetooth headphones with noise cancellation and 30-hour battery life"},
    {"description": "Organic cotton t-shirt in navy blue, crew neck, breathable fabric for summer"},
    {"description": "Stainless steel water bottle, 32oz capacity, keeps drinks cold for 24 hours"},
    {"description": "Ergonomic office chair with lumbar support and adjustable armrests"},
    {"description": "Smart LED light bulb compatible with Alexa and Google Home, color changing"},
    {"description": "Yoga mat with extra thickness, non-slip surface, includes carrying strap"},
    {"description": "Portable charger 20000mAh with fast charging and dual USB ports"},
    {"description": "Coffee maker with programmable timer and thermal carafe, 12-cup capacity"},
    {"description": "Running shoes with cushioned sole and breathable mesh upper for men"},
    {"description": "Laptop backpack with USB charging port and water-resistant material"},
    {"description": "Cast iron skillet 12-inch pre-seasoned for cooking and baking"},
    {"description": "Bamboo cutting board set with juice groove and hanging hole"},
    {"description": "Electric toothbrush with pressure sensor and multiple cleaning modes"},
    {"description": "Memory foam pillow with cooling gel layer for side and back sleepers"},
    {"description": "Fitness tracker watch with heart rate monitor and sleep tracking"},
    {"description": "Bluetooth speaker waterproof with 360-degree sound and 12-hour playtime"},
    {"description": "Adjustable dumbbell set from 5 to 52.5 pounds for home gym"},
    {"description": "Silk pillowcase hypoallergenic reduces hair frizz and skin wrinkles"},
    {"description": "Digital air fryer 6-quart capacity with 8 preset cooking functions"},
    {"description": "Gaming mouse with RGB lighting and programmable buttons, 16000 DPI"},
    {"description": "Leather wallet with RFID blocking technology and coin pocket"},
    {"description": "Indoor security camera with night vision and two-way audio"},
    {"description": "Electric kettle with temperature control for tea and coffee brewing"},
    {"description": "Resistance bands set with 5 different strength levels and door anchor"},
    {"description": "Sunglasses with polarized lenses and UV400 protection for driving"},
    {"description": "Blender with 1000-watt motor for smoothies and crushing ice"},
    {"description": "Mechanical keyboard with blue switches and RGB backlighting"},
    {"description": "Bath towel set 100% cotton ultra soft and absorbent 6-piece"},
    {"description": "Desk lamp with adjustable brightness levels and USB charging port"},
    {"description": "Winter jacket with down insulation and waterproof outer shell"},
    {"description": "Smartphone gimbal stabilizer for smooth video recording"},
    {"description": "Air purifier with HEPA filter removes 99.97% of allergens and dust"},
    {"description": "Slow cooker 6-quart programmable with timer and keep-warm function"},
    {"description": "Hiking boots waterproof with ankle support and Vibram sole"},
    {"description": "Wireless doorbell with video camera and motion detection alerts"},
    {"description": "Electric shaver for men with pop-up trimmer and wet/dry use"},
    {"description": "Jewelry organizer with multiple compartments and mirror"},
    {"description": "Cordless vacuum cleaner lightweight with LED display and HEPA filtration"},
    {"description": "Standing desk converter adjustable height for ergonomic workspace"},
    {"description": "Instant pot pressure cooker 8-quart with 14 cooking programs"},
    {"description": "Noise machine with 20 soothing sounds for better sleep"},
    {"description": "Trail running shoes with aggressive tread pattern and rock plate"},
    {"description": "Smart thermostat with learning capability and energy-saving features"},
    {"description": "Electric wine opener with foil cutter and rechargeable battery"},
    {"description": "Weighted blanket 15 pounds for adults reduces anxiety and improves sleep"},
    {"description": "Drone with 4K camera and GPS auto-return for aerial photography"},
    {"description": "Stainless steel cookware set 10-piece with tempered glass lids"},
    {"description": "Essential oil diffuser ultrasonic with color-changing LED lights"},
    {"description": "Garden hose expandable 100 feet with spray nozzle included"},
    {"description": "Tablet stand adjustable aluminum for desk and kitchen use"}
]

movies_books_dataset = [
    {"description": "Epic space opera following a rebellion against an oppressive galactic empire"},
    {"description": "Psychological thriller about a detective hunting a serial killer in a rainy city"},
    {"description": "Coming-of-age story set in 1950s America exploring themes of innocence and identity"},
    {"description": "Fantasy adventure with wizards, dragons, and a quest to destroy an ancient artifact"},
    {"description": "Romantic comedy about two coworkers falling in love during a business trip to Paris"},
    {"description": "Historical fiction depicting life during World War II from a child's perspective"},
    {"description": "Science fiction novel exploring artificial intelligence and what it means to be human"},
    {"description": "Mystery novel set in an English manor where guests are murdered one by one"},
    {"description": "Biography of a civil rights leader who changed the course of American history"},
    {"description": "Horror film about a haunted house that traps a family inside"},
    {"description": "Action movie featuring a spy on a mission to prevent global catastrophe"},
    {"description": "Animated film about emotions living inside a young girl's mind"},
    {"description": "Self-help book offering strategies for personal productivity and success"},
    {"description": "Dystopian novel where society is divided into factions based on virtues"},
    {"description": "True crime documentary investigating an unsolved murder from the 1990s"},
    {"description": "Musical drama following aspiring artists pursuing their dreams in Los Angeles"},
    {"description": "Cookbook featuring authentic Italian recipes passed down through generations"},
    {"description": "Memoir of a woman's solo journey hiking the Pacific Crest Trail"},
    {"description": "Political thriller about conspiracy within the highest levels of government"},
    {"description": "Children's book teaching valuable lessons about friendship and kindness"},
    {"description": "Western film set in the 1880s about an outlaw seeking redemption"},
    {"description": "Philosophy book examining existentialism and the meaning of life"},
    {"description": "Superhero movie where heroes must unite to defeat an alien invasion"},
    {"description": "Poetry collection exploring themes of love, loss, and nature"},
    {"description": "Business book analyzing successful companies and their strategies"},
    {"description": "Adventure novel about treasure hunters searching for lost Incan gold"},
    {"description": "Documentary about climate change and its impact on polar ecosystems"},
    {"description": "Sports biography chronicling the career of a legendary basketball player"},
    {"description": "Gothic romance set in Victorian England with mysterious secrets"},
    {"description": "Cookbook with healthy plant-based recipes for every meal"},
    {"description": "War epic depicting the D-Day invasion during World War II"},
    {"description": "Financial advice book teaching principles of wealth building and investing"},
    {"description": "Fantasy series about a young orphan discovering magical powers"},
    {"description": "Travel memoir documenting a year living in rural Japan"},
    {"description": "Crime drama series following detectives solving cold cases"},
    {"description": "Parenting guide with research-based strategies for raising confident children"},
    {"description": "Historical drama about the life of a famous renaissance painter"},
    {"description": "Science book explaining quantum physics for general audiences"},
    {"description": "Thriller novel about a lawyer defending an innocent man on death row"},
    {"description": "Romantic drama spanning decades following two lovers separated by war"},
    {"description": "Graphic novel depicting a superhero's origin story and first adventures"},
    {"description": "Psychology book exploring how habits form and how to change them"},
    {"description": "Adventure film about explorers discovering a hidden island with dinosaurs"},
    {"description": "Literary fiction examining family dynamics across three generations"},
    {"description": "Fitness book with workout plans and nutritional guidance"},
    {"description": "Horror anthology series with different scary stories each episode"},
    {"description": "Economics book analyzing global markets and trade policies"},
    {"description": "Drama about a teacher inspiring students in an underprivileged school"},
    {"description": "Art history book surveying movements from impressionism to modern art"},
    {"description": "Comedy series following the misadventures of coworkers in an office"}
]

documents_dataset = [
    {"description": "Research paper on machine learning applications in healthcare diagnostics"},
    {"description": "Technical documentation for REST API authentication and authorization protocols"},
    {"description": "White paper analyzing cybersecurity threats in cloud computing environments"},
    {"description": "Legal contract outlining terms and conditions for software licensing agreements"},
    {"description": "Medical study on the effectiveness of new treatment for type 2 diabetes"},
    {"description": "Marketing report analyzing consumer behavior trends in e-commerce platforms"},
    {"description": "Engineering specification for bridge construction and load-bearing requirements"},
    {"description": "Academic thesis exploring renewable energy solutions for urban areas"},
    {"description": "Policy document detailing data privacy regulations and compliance requirements"},
    {"description": "Financial report showing quarterly earnings and revenue projections"},
    {"description": "Tutorial guide for beginners learning Python programming fundamentals"},
    {"description": "Scientific article about climate patterns and ocean temperature changes"},
    {"description": "User manual for operating industrial machinery and safety protocols"},
    {"description": "Case study examining successful digital transformation in retail industry"},
    {"description": "Educational curriculum for teaching mathematics to middle school students"},
    {"description": "Architecture blueprint for sustainable residential building design"},
    {"description": "News article covering recent developments in space exploration technology"},
    {"description": "Employee handbook outlining company policies and workplace procedures"},
    {"description": "Research findings on the psychological effects of social media usage"},
    {"description": "Technical specification for 5G network infrastructure deployment"},
    {"description": "Grant proposal for funding community development projects"},
    {"description": "Installation guide for configuring database management systems"},
    {"description": "Environmental impact assessment for proposed highway construction"},
    {"description": "Patent application describing innovative battery technology design"},
    {"description": "Training materials for customer service representatives handling complaints"},
    {"description": "Statistical analysis of voting patterns in recent elections"},
    {"description": "Software requirements document for mobile application development"},
    {"description": "Medical guidelines for diagnosing and treating cardiovascular disease"},
    {"description": "Market research report on emerging technologies in automotive industry"},
    {"description": "Safety inspection checklist for manufacturing facilities"},
    {"description": "Academic paper investigating effects of sleep deprivation on cognition"},
    {"description": "Project proposal for implementing new inventory management system"},
    {"description": "Regulatory compliance document for pharmaceutical drug approval"},
    {"description": "Troubleshooting guide for resolving network connectivity issues"},
    {"description": "Strategic business plan for expanding into international markets"},
    {"description": "Clinical trial results for experimental cancer treatment protocol"},
    {"description": "Technical manual for aircraft maintenance and repair procedures"},
    {"description": "Economic forecast predicting inflation rates and GDP growth"},
    {"description": "Security audit report identifying vulnerabilities in web applications"},
    {"description": "Lesson plan for teaching creative writing to high school students"},
    {"description": "Geological survey documenting mineral deposits in mountain regions"},
    {"description": "Quality assurance procedures for food safety in restaurant operations"},
    {"description": "Migration guide for upgrading legacy systems to cloud infrastructure"},
    {"description": "Anthropological study of cultural practices in indigenous communities"},
    {"description": "Standard operating procedures for emergency response protocols"},
    {"description": "Investment analysis evaluating risk factors in stock portfolios"},
    {"description": "User experience research report on mobile app interface design"},
    {"description": "Agricultural guide for organic farming techniques and pest management"},
    {"description": "Audit report examining financial statements for accounting accuracy"},
    {"description": "Performance review template for evaluating employee productivity metrics"}
]

customer_reviews_dataset = [
    {"description": "Amazing product! Exceeded my expectations. Fast shipping and great quality."},
    {"description": "Disappointed with the purchase. Item arrived damaged and customer service was unhelpful."},
    {"description": "Decent product for the price. Not the best quality but gets the job done."},
    {"description": "Absolutely love it! Have been using it daily for months with no issues."},
    {"description": "Would not recommend. Stopped working after just two weeks of use."},
    {"description": "Great value for money. Exactly as described and arrived on time."},
    {"description": "The quality is questionable. Feels cheap and flimsy compared to similar products."},
    {"description": "Best purchase I've made this year! Cannot believe how well this works."},
    {"description": "Okay product but shipping took forever and packaging was poor."},
    {"description": "Highly recommend! My friends are now buying this after seeing mine."},
    {"description": "Total waste of money. Does not work as advertised at all."},
    {"description": "Good product overall with minor flaws. Customer service was responsive."},
    {"description": "Instructions were confusing but once I figured it out, works perfectly."},
    {"description": "Not worth the premium price. You can find better alternatives cheaper."},
    {"description": "Fantastic! Solved my problem immediately. Wish I had bought this sooner."},
    {"description": "Received wrong item initially but company quickly sent replacement."},
    {"description": "Product is fine but took weeks longer than estimated delivery date."},
    {"description": "Incredible quality and attention to detail. Worth every penny."},
    {"description": "Broke after one use. Very frustrating and disappointing experience."},
    {"description": "Solid product that does what it promises. No complaints here."},
    {"description": "Color was different than shown in photos. Otherwise satisfied."},
    {"description": "Outstanding customer support helped me resolve my issue quickly."},
    {"description": "Cheap material that looks nothing like the product description."},
    {"description": "Better than expected! Working great for my specific needs."},
    {"description": "Instructions missing and difficult to assemble without guidance."},
    {"description": "Five stars! This product has changed my daily routine for the better."},
    {"description": "Average product with nothing special to distinguish it from competitors."},
    {"description": "Excellent build quality and design. Very impressed with this purchase."},
    {"description": "Smelled terrible when I opened the package. Had to air it out for days."},
    {"description": "Perfect size and exactly what I was looking for. Very happy!"},
    {"description": "Customer service refused to honor the warranty. Very disappointed."},
    {"description": "Nice product but slightly smaller than I expected from the dimensions."},
    {"description": "Game changer! Cannot imagine going back to my old one."},
    {"description": "Parts don't fit together properly. Design flaw seems obvious."},
    {"description": "Reliable and consistent performance every time I use it."},
    {"description": "Packaging was excessive but product inside was in perfect condition."},
    {"description": "Not as durable as I hoped. Showing signs of wear already."},
    {"description": "Impressive technology at an affordable price point. Great deal!"},
    {"description": "Difficult to clean and maintain. Wish I had known before buying."},
    {"description": "Stylish design that looks great in my home. Functionality is good too."},
    {"description": "Arrived with missing pieces. Still waiting on replacement parts."},
    {"description": "Comfortable and well-made. Would definitely purchase again."},
    {"description": "Noisy operation makes it annoying to use during quiet hours."},
    {"description": "Exactly as advertised. No surprises, which is refreshing nowadays."},
    {"description": "Battery life is much shorter than claimed in specifications."},
    {"description": "Versatile product with many uses. Getting more value than expected."},
    {"description": "Returned immediately. Quality was unacceptable for this price range."},
    {"description": "Simple to use right out of the box. Very intuitive interface."},
    {"description": "Heavy and bulky. Not as portable as I thought it would be."},
    {"description": "Perfect gift! Recipient was thrilled and uses it constantly."}
]

customer_reviews_dataset = [
    {"description": "Amazing product! Exceeded my expectations. Fast shipping and great quality."},
    {"description": "Disappointed with the purchase. Item arrived damaged and customer service was unhelpful."},
    {"description": "Decent product for the price. Not the best quality but gets the job done."},
    {"description": "Absolutely love it! Have been using it daily for months with no issues."},
    {"description": "Would not recommend. Stopped working after just two weeks of use."},
    {"description": "Great value for money. Exactly as described and arrived on time."},
    {"description": "The quality is questionable. Feels cheap and flimsy compared to similar products."},
    {"description": "Best purchase I've made this year! Cannot believe how well this works."},
    {"description": "Okay product but shipping took forever and packaging was poor."},
    {"description": "Highly recommend! My friends are now buying this after seeing mine."},
    {"description": "Total waste of money. Does not work as advertised at all."},
    {"description": "Good product overall with minor flaws. Customer service was responsive."},
    {"description": "Instructions were confusing but once I figured it out, works perfectly."},
    {"description": "Not worth the premium price. You can find better alternatives cheaper."},
    {"description": "Fantastic! Solved my problem immediately. Wish I had bought this sooner."},
    {"description": "Received wrong item initially but company quickly sent replacement."},
    {"description": "Product is fine but took weeks longer than estimated delivery date."},
    {"description": "Incredible quality and attention to detail. Worth every penny."},
    {"description": "Broke after one use. Very frustrating and disappointing experience."},
    {"description": "Solid product that does what it promises. No complaints here."},
    {"description": "Color was different than shown in photos. Otherwise satisfied."},
    {"description": "Outstanding customer support helped me resolve my issue quickly."},
    {"description": "Cheap material that looks nothing like the product description."},
    {"description": "Better than expected! Working great for my specific needs."},
    {"description": "Instructions missing and difficult to assemble without guidance."},
    {"description": "Five stars! This product has changed my daily routine for the better."},
    {"description": "Average product with nothing special to distinguish it from competitors."},
    {"description": "Excellent build quality and design. Very impressed with this purchase."},
    {"description": "Smelled terrible when I opened the package. Had to air it out for days."},
    {"description": "Perfect size and exactly what I was looking for. Very happy!"},
    {"description": "Customer service refused to honor the warranty. Very disappointed."},
    {"description": "Nice product but slightly smaller than I expected from the dimensions."},
    {"description": "Game changer! Cannot imagine going back to my old one."},
    {"description": "Parts don't fit together properly. Design flaw seems obvious."},
    {"description": "Reliable and consistent performance every time I use it."},
    {"description": "Packaging was excessive but product inside was in perfect condition."},
    {"description": "Not as durable as I hoped. Showing signs of wear already."},
    {"description": "Impressive technology at an affordable price point. Great deal!"},
    {"description": "Difficult to clean and maintain. Wish I had known before buying."},
    {"description": "Stylish design that looks great in my home. Functionality is good too."},
    {"description": "Arrived with missing pieces. Still waiting on replacement parts."},
    {"description": "Comfortable and well-made. Would definitely purchase again."},
    {"description": "Noisy operation makes it annoying to use during quiet hours."},
    {"description": "Exactly as advertised. No surprises, which is refreshing nowadays."},
    {"description": "Battery life is much shorter than claimed in specifications."},
    {"description": "Versatile product with many uses. Getting more value than expected."},
    {"description": "Returned immediately. Quality was unacceptable for this price range."},
    {"description": "Simple to use right out of the box. Very intuitive interface."},
    {"description": "Heavy and bulky. Not as portable as I thought it would be."},
    {"description": "Perfect gift! Recipient was thrilled and uses it constantly."}
]


# Combined dataset for diverse testing
your_dataset = product_dataset + movies_books_dataset + documents_dataset + customer_reviews_dataset
print(f"Total items: {len(your_dataset)}")  # 200 items


Total items: 200


In [5]:
def upload_with_timing(collection_name, data, config_name):
    embeddings = encoder.encode([d["description"] for d in data], show_progress_bar=True).tolist()

    points = []
    for i, item in enumerate(data):
        embedding = embeddings[i]

        points.append(
            models.PointStruct(
                id=i,
                vector=embedding,
                payload={
                    **item,
                    "length": len(item["description"]),
                    "word_count": len(item["description"].split()),
                    "has_keywords": any(
                        keyword in item["description"].lower() for keyword in ["important", "key", "main"]
                    ),
                },
            )
        )

    # Warmup
    client.query_points(collection_name=collection_name, query=points[0].vector, limit=1)

    start_time = time.time()
    client.upload_points(collection_name=collection_name, points=points)
    upload_time = time.time() - start_time

    print(f"{config_name}: Uploaded {len(points)} points in {upload_time:.2f}s")
    return upload_time


# Load your dataset here. The larger the dataset, the more accurate the benchmark will be.
# your_dataset = [{"description": "This is a description of a product"}, ...]

# Upload to each collection
upload_times = {}
for config in configs:
    collection_name = f"my_domain_{config['name']}"
    upload_times[config["name"]] = upload_with_timing(collection_name, your_dataset, config["name"])


def wait_for_indexing(collection_name, timeout=60, poll_interval=1):
    print(f"Waiting for collection '{collection_name}' to be indexed...")
    start_time = time.time()

    while time.time() - start_time < timeout:
        info = client.get_collection(collection_name=collection_name)

        if info.indexed_vectors_count > 0 and info.status == models.CollectionStatus.GREEN:
            print(f"Success! Collection '{collection_name}' is indexed and ready.")
            print(f" - Status: {info.status.value}")
            print(f" - Indexed vectors: {info.indexed_vectors_count}")
            return

        print(f" - Status: {info.status.value}, Indexed vectors: {info.indexed_vectors_count}. Waiting...")
        time.sleep(poll_interval)

    info = client.get_collection(collection_name=collection_name)
    raise Exception(
        f"Timeout reached after {timeout} seconds. Collection '{collection_name}' is not ready. "
        f"Final status: {info.status.value}, Indexed vectors: {info.indexed_vectors_count}"
    )


for config in configs:
    if config["m"] > 0:  # m=0 has no HNSW to wait for
        collection_name = f"my_domain_{config['name']}"
        wait_for_indexing(collection_name)

Batches: 100%|██████████| 7/7 [00:00<00:00, 17.49it/s]


fast_initial_upload: Uploaded 200 points in 3.65s


Batches: 100%|██████████| 7/7 [00:00<00:00, 23.89it/s]


memory_optimized: Uploaded 200 points in 9.39s


Batches: 100%|██████████| 7/7 [00:00<00:00, 16.72it/s]


balanced: Uploaded 200 points in 2.66s


Batches: 100%|██████████| 7/7 [00:00<00:00, 17.40it/s]


high_quality: Uploaded 200 points in 3.43s
Waiting for collection 'my_domain_memory_optimized' to be indexed...
Success! Collection 'my_domain_memory_optimized' is indexed and ready.
 - Status: green
 - Indexed vectors: 200
Waiting for collection 'my_domain_balanced' to be indexed...
Success! Collection 'my_domain_balanced' is indexed and ready.
 - Status: green
 - Indexed vectors: 200
Waiting for collection 'my_domain_high_quality' to be indexed...
Success! Collection 'my_domain_high_quality' is indexed and ready.
 - Status: green
 - Indexed vectors: 200


## Step 4: Benchmark Search Performance

In [10]:
def benchmark_search(collection_name, query_embedding, ef_values=[64, 128, 256]):
    # Warmup
    client.query_points(collection_name=collection_name, query=query_embedding, limit=1)

    # hnsw_ef: higher = better recall, but slower. Tune per your latency goal.
    results = {}
    for hnsw_ef in ef_values:
        times = []

        # Run multiple queries for more reliable timing
        for _ in range(25):
            start_time = time.time()

            _ = client.query_points(
                collection_name=collection_name,
                query=query_embedding,
                limit=10,
                search_params=models.SearchParams(hnsw_ef=hnsw_ef),
                with_payload=False,
            )

            times.append((time.time() - start_time) * 1000)

        results[hnsw_ef] = {
            "avg_time": np.mean(times),
            "min_time": np.min(times),
            "max_time": np.max(times),
        }

    return results


test_query = "wireless headphones with noise cancellation"
query_embedding = encoder.encode(test_query).tolist()

performance_results = {}
for config in configs:
    if config["m"] > 0:  # Skip m=0 collections for search
        collection_name = f"my_domain_{config['name']}"
        performance_results[config["name"]] = benchmark_search(
            collection_name, query_embedding
        )

print("\nPerformance Results (in milliseconds):")



Performance Results (in milliseconds):


In [14]:
response = client.query_points(
    collection_name=f"my_domain_{configs[1]['name']}",
    query=query_embedding,
    limit=5,
    with_payload=True
)

print(f"\nTop 10 Results for: '{test_query}'\n")
print("=" * 80)

for i, point in enumerate(response.points, 1):
    print(f"Rank {i}:")
    print(f"  ID: {point.id}")
    print(f"  Score: {point.score:.4f}")
    print(f"  Description: {point.payload['description']}")
    print()



Top 10 Results for: 'wireless headphones with noise cancellation'

Rank 1:
  ID: 0
  Score: 0.7843
  Description: Wireless Bluetooth headphones with noise cancellation and 30-hour battery life

Rank 2:
  ID: 40
  Score: 0.3851
  Description: Noise machine with 20 soothing sounds for better sleep

Rank 3:
  ID: 192
  Score: 0.3300
  Description: Noisy operation makes it annoying to use during quiet hours.

Rank 4:
  ID: 15
  Score: 0.2863
  Description: Bluetooth speaker waterproof with 360-degree sound and 12-hour playtime

Rank 5:
  ID: 20
  Score: 0.2832
  Description: Leather wallet with RFID blocking technology and coin pocket



## Step 5: Measure Payload Indexing Impact

In [8]:
def test_filtering_performance(collection_name):
    query_embedding = encoder.encode("your filter test query").tolist()

    # Test filter without index
    filter_condition = models.Filter(
        must=[models.FieldCondition(key="length", range=models.Range(gte=10, lte=200))]
    )

    # Demo only: unindexed_filtering_retrieve=True forces a scan; turn it off right after measuring.
    client.update_collection(
        collection_name=collection_name,
        strict_mode_config=models.StrictModeConfig(unindexed_filtering_retrieve=True),
    )

    # Warmup
    client.query_points(collection_name=collection_name, query=query_embedding, limit=1)

    # Timing without payload index
    times = []
    for _ in range(25):
        start_time = time.time()
        _ = client.query_points(
            collection_name=collection_name,
            query=query_embedding,
            query_filter=filter_condition,
            limit=10,
            with_payload=False,
        )
        times.append((time.time() - start_time) * 1000)
    time_without_index = np.mean(times)

    # Create payload index
    client.create_payload_index(
        collection_name=collection_name,
        field_name="length",
        field_schema=models.PayloadSchemaType.INTEGER,
        wait=True,
    )

    # HNSW was already built; adding the payload index doesn’t rebuild it.
    # Bump ef_construct (+1) once to trigger a safe rebuild.
    base_ef = client.get_collection(
        collection_name=collection_name
    ).config.hnsw_config.ef_construct
    new_ef_construct = base_ef + 1

    client.update_collection(
        collection_name=collection_name,
        hnsw_config=models.HnswConfigDiff(ef_construct=new_ef_construct),
        strict_mode_config=models.StrictModeConfig(
            unindexed_filtering_retrieve=False
        ),  # Turn off scanning and use payload index instead.
    )

    wait_for_indexing(collection_name)

    # Warmup
    client.query_points(collection_name=collection_name, query=query_embedding, limit=1)

    # Timing with index
    times = []
    for _ in range(25):
        start_time = time.time()
        _ = client.query_points(
            collection_name=collection_name,
            query=query_embedding,
            query_filter=filter_condition,
            limit=10,
            with_payload=False,
        )
        times.append((time.time() - start_time) * 1000)
    time_with_index = np.mean(times)

    return {
        "without_index": time_without_index,
        "with_index": time_with_index,
        "speedup": time_without_index / time_with_index,
    }


# Test on your best performing collection
best_collection = "my_domain_balanced"  # Choose based on your results
filtering_results = test_filtering_performance(best_collection)

Waiting for collection 'my_domain_balanced' to be indexed...
Success! Collection 'my_domain_balanced' is indexed and ready.
 - Status: green
 - Indexed vectors: 200


## Step 6: Analyze Your Results

In [9]:
print("=" * 60)
print("PERFORMANCE OPTIMIZATION RESULTS")
print("=" * 60)

print("\n1) Upload Performance:")
for config_name, time_taken in upload_times.items():
    print(f"   {config_name}: {time_taken:.2f}s")

print("\n2) Search Performance (hnsw_ef=128):")
for config_name, results in performance_results.items():
    if 128 in results:
        print(f"   {config_name}: {results[128]['avg_time']:.2f}ms")

print("\n3) Filtering Impact:")
print(f"   Without index: {filtering_results['without_index']:.2f}ms")
print(f"   With index: {filtering_results['with_index']:.2f}ms")
print(f"   Speedup: {filtering_results['speedup']:.1f}x")

PERFORMANCE OPTIMIZATION RESULTS

1) Upload Performance:
   fast_initial_upload: 3.65s
   memory_optimized: 9.39s
   balanced: 2.66s
   high_quality: 3.43s

2) Search Performance (hnsw_ef=128):
   memory_optimized: 173.04ms
   balanced: 187.90ms
   high_quality: 172.46ms

3) Filtering Impact:
   Without index: 187.02ms
   With index: 186.00ms
   Speedup: 1.0x
